# Repeatability and direction

Bounded gain concentration, sampling and BTC-direction diagnostics. These features remain diagnostic.

In [ ]:
from pathlib import Path
import sys
import json
import math
import hashlib
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd() / "scratchpad/hyperliquid-ic" if (Path.cwd() / "scratchpad/hyperliquid-ic").exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / "_artifacts-stable-profit"
ARTIFACTS.mkdir(exist_ok=True)
REWRITE = PROJECT_DIR / "_artifacts-rewrite"
from ic_research import ResearchConfig, completed_daily_window, simulate_allocator, summarise_backtest, daily_feature_ic, write_json
from stable_profit import classification_metrics, paired_block_sharpe_samples, short_core_columns, simulate_stable_policy, walk_forward_loss_predictions, weekly_curve

CONFIG = ResearchConfig()
features = pd.read_parquet(REWRITE / "features.parquet")
labels = pd.read_parquet(REWRITE / "labels.parquet")
observations = pd.read_parquet(REWRITE / "observations.parquet")
evaluation_start, evaluation_end = completed_daily_window(observations)
features = features[features.date.between(evaluation_start - pd.Timedelta(days=CONFIG.training_history_days), evaluation_end)].copy()
labels = labels[labels.date.isin(features.date)].copy()
evaluation = features[features.date.between(evaluation_start, evaluation_end)].copy()
print("Stable-profit panel:", features.shape, "| dates:", features.date.nunique(), "| evaluation:", evaluation_start, "to", evaluation_end)


In [ ]:
obs = observations.sort_values(["address", "timestamp"]).copy()
obs["prev_timestamp"] = obs.groupby("address").timestamp.shift()
obs["prev_price"] = obs.groupby("address").share_price.shift()
obs["duration_days"] = (obs.timestamp - obs.prev_timestamp).dt.total_seconds() / 86400
obs["log_return"] = np.log(obs.share_price / obs.prev_price)
obs = obs[obs.duration_days.gt(0)].copy()
by_address = {address: group.reset_index(drop=True) for address, group in obs.groupby("address", sort=False)}
rows = []
for row in features.itertuples(index=False):
    history = by_address.get(row.address)
    if history is None:
        continue
    end = pd.Timestamp(row.last_observation_ts)
    values = history[history.timestamp <= end]
    item = {"date": row.date, "address": row.address}
    for window in (30, 60):
        subset = values[values.timestamp > end - pd.Timedelta(days=window)]
        returns = subset.log_return.to_numpy(dtype=float)
        positive = returns[returns > 0]
        total = positive.sum()
        item[f"gain_share_{window}"] = float(positive.max() / total) if len(positive) and total > 0 else (0.0 if len(positive) == 0 and len(returns) >= 4 else np.nan)
        item[f"top3_gain_share_{window}"] = float(np.sort(positive)[-3:].sum() / total) if len(positive) >= 4 and total > 0 else np.nan
        item[f"growth_ex_best_{window}"] = float(returns.sum() - (positive.max() if len(positive) else 0)) if len(returns) else np.nan
        item[f"negative_fraction_{window}"] = float(np.mean(returns < 0)) if len(returns) >= 4 else np.nan
        item[f"interval_count_{window}"] = int(len(returns))
        item[f"covered_days_{window}"] = float(subset.duration_days.sum()) if len(returns) else 0.0
    rows.append(item)
bounded = pd.DataFrame(rows)
bounded.to_parquet(ARTIFACTS / "bounded-path-features.parquet", index=False)
print("Bounded path diagnostics:", bounded.shape)


In [ ]:
merged = features[["date", "address", "tvl_current", "observed_interval_count_30", "interval_downside_dev_30", "daily_vol_30", "btc_beta_30"]].merge(bounded, on=["date", "address"], how="inner").merge(labels[["date", "address", "forward_log_growth_raw_30"]], on=["date", "address"], how="inner")
merged = merged[merged.date.between(evaluation_start, evaluation_end)].copy()
rows = []
for feature in ["gain_share_30", "top3_gain_share_30", "growth_ex_best_30", "negative_fraction_30", "btc_beta_30", "gain_share_60", "top3_gain_share_60", "growth_ex_best_60", "negative_fraction_60"]:
    data = merged[["date", "address", feature, "forward_log_growth_raw_30", "observed_interval_count_30"]].rename(columns={"forward_log_growth_raw_30": "target"})
    data["eligible"] = True
    ic = daily_feature_ic(data, data, feature=feature, target="target", sign=1.0)
    if not ic.empty:
        rows.append({"feature": feature, "dates": len(ic), "mean_rank_ic": ic.rank_ic.mean(), "median_rank_ic": ic.rank_ic.median(), "positive_date_fraction": float((ic.rank_ic > 0).mean())})
diagnostic_ic = pd.DataFrame(rows).sort_values("mean_rank_ic", ascending=False)
diagnostic_ic.to_csv(ARTIFACTS / "bounded-diagnostic-ic.csv", index=False)
display(diagnostic_ic)


In [ ]:
base_index = features.set_index(["date", "address"])
merged["available_history_days"] = base_index.reindex(merged.set_index(["date", "address"]).index).available_history_days.to_numpy()
merged["btc_beta_abs"] = merged.btc_beta_30.abs()
merged["history_cohort"] = pd.cut(merged.available_history_days, [-np.inf, 30, 90, 180, 360, np.inf], labels=["<30", "30-89", "90-179", "180-359", "360+"], right=False)
cohorts = merged.groupby("history_cohort", observed=False).agg(rows=("address", "size"), mean_growth=("forward_log_growth_raw_30", "mean"), loss_probability=("forward_log_growth_raw_30", lambda s: float((s < math.log(.9)).mean())), mean_beta_abs=("btc_beta_abs", "mean")).reset_index()
cohorts.to_csv(ARTIFACTS / "direction-sampling-cohorts.csv", index=False)
family_columns = short_core_columns(features) + [c for c in ["gain_share_30", "top3_gain_share_30", "growth_ex_best_30", "negative_fraction_30"] if c in bounded]
family_features = features.merge(bounded, on=["date", "address"], how="left")
family_predictions = walk_forward_loss_predictions(family_features, labels, feature_columns=family_columns, horizon=30, evaluation_dates=pd.DatetimeIndex(sorted(evaluation.date.unique())), min_train_dates=60, c=1.0)
family_predictions.to_parquet(ARTIFACTS / "family-loss-predictions-30.parquet", index=False)
write_json(ARTIFACTS / "family-loss-classifier-metrics.json", classification_metrics(family_predictions))
display(cohorts)
